## 1. 라이브러리 임포트 및 데이터 로드

In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import IsolationForest
from imblearn.over_sampling import SMOTE

df = pd.read_csv('../data/online_shoppers_intention.csv')

## 2. 인코딩 (Bool / LabelEncoder)
범주형 변수(Month, VisitorType) 숫자 변환, 불리언타입 0/1 변환

In [2]:
# 1. bool 타입 → 0/1 변환
df['Weekend'] = df['Weekend'].astype(int)
df['Revenue'] = df['Revenue'].astype(int)

# 2. 범주형 → LabelEncoder
le = LabelEncoder()
df['Month'] = le.fit_transform(df['Month'])
df['VisitorType'] = le.fit_transform(df['VisitorType'])

print(df.dtypes)
print(df.head())

Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                        int64
OperatingSystems             int64
Browser                      int64
Region                       int64
TrafficType                  int64
VisitorType                  int64
Weekend                      int64
Revenue                      int64
dtype: object
   Administrative  Administrative_Duration  Informational  \
0               0                      0.0              0   
1               0                      0.0              0   
2               0                      0.0              0   
3               0                      0.0              0   
4               0                      0.0         

## 4. 이상치 제거 (IsolationForest) 및 정규화 (StandardScaler) 
contamination=0.01로 설정해 전체 데이터의 1%를 이상치로 간주하고 제거한다.

In [3]:
# 3. X, y 분리
X = df.drop('Revenue', axis=1)
y = df['Revenue']

# 4. IsolationForest로 이상치 제거
iso = IsolationForest(contamination=0.01, random_state=42)
mask = iso.fit_predict(X)
X = X[mask == 1]
y = y[mask == 1]
print(f'이상치 제거 후 데이터 크기: {X.shape}')

# 5. StandardScaler로 정규화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6. SMOTE로 클래스 불균형 처리
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_scaled, y)
print(f'SMOTE 적용 후 데이터 크기: {X_resampled.shape}')
print(f'클래스 분포:\n{pd.Series(y_resampled).value_counts()}')

이상치 제거 후 데이터 크기: (12206, 17)
SMOTE 적용 후 데이터 크기: (20658, 17)
클래스 분포:
Revenue
0    10329
1    10329
Name: count, dtype: int64


## 6. 클래스 불균형 처리 (SMOTE)
원래 데이터는 구매(1): 15%, 미구매(0): 85%로 불균형하다.
SMOTE로 소수 클래스를 오버샘플링해 균형을 맞춘다.

In [4]:
np.save('../data/X_resampled.npy', X_resampled)
np.save('../data/y_resampled.npy', y_resampled)
print('저장 완료!')

저장 완료!
